# GxP-LLM quantization and evaluation

Attach `gxp-source`, `gxp-data`, and the output of a completed fine-tuning notebook. This notebook publishes GPTQ and AWQ models plus a complete, comparable evaluation for each variant.

In [ ]:
%pip install -q transformers accelerate auto-gptq autoawq sentence-transformers rouge-score nltk litellm wandb peft bitsandbytes

import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
print(torch.cuda.get_device_name(0))

In [ ]:
import gc, json, os, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import wandb

SOURCE_DIR, DATA_DIR = Path('/kaggle/input/gxp-source'), Path('/kaggle/input/gxp-data')
# Replace this with the actual notebook-output attachment slug shown in Kaggle's Input pane.
MODEL_PATH = Path('/kaggle/input/gxp-finetune-output/gxp_train/merged_16bit')
assert (SOURCE_DIR / 'eval' / 'run.py').exists() and (DATA_DIR / 'train.jsonl').exists() and MODEL_PATH.exists(), 'Attach source, data, and fine-tuning notebook output.'
sys.path.insert(0, str(SOURCE_DIR))
OUTPUT_DIR = Path('/kaggle/working/gxp_quantization'); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JUDGE_MODEL = 'gemini/gemini-2.5-flash'

def load_secret(name):
    value = UserSecretsClient().get_secret(name)
    if not value: raise RuntimeError(f'Add {name} as a Kaggle Secret.')
    os.environ[name] = value
load_secret('WANDB_API_KEY')
load_secret({'gemini/': 'GEMINI_API_KEY', 'groq/': 'GROQ_API_KEY', 'nvidia_nim/': 'NVIDIA_API_KEY'}[next(k for k in ('gemini/', 'groq/', 'nvidia_nim/') if JUDGE_MODEL.startswith(k))])
config = {'stage':'quantization','source_model':str(MODEL_PATH),'bits':4,'group_size':128,'judge_model':JUDGE_MODEL}
run = wandb.init(project='gxp-llm', name='quantize-4bit', job_type='quantize', tags=['gptq','awq'], config=config)
(OUTPUT_DIR / 'experiment_config.json').write_text(json.dumps(config, indent=2))

def calibration_texts(limit=128):
    examples = [json.loads(line) for line in (DATA_DIR / 'train.jsonl').read_text().splitlines()[:limit]]
    return [x['messages'][0]['content'] + '\n' + x['messages'][1]['content'] for x in examples]
CALIBRATION_TEXTS = calibration_texts()

In [ ]:
# GPTQ 4-bit
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True); tokenizer.pad_token = tokenizer.eos_token
gptq_path = OUTPUT_DIR / 'gptq-4bit'
gptq_model = AutoGPTQForCausalLM.from_pretrained(str(MODEL_PATH), quantize_config=BaseQuantizeConfig(bits=4, group_size=128, desc_act=False, sym=True), device_map='auto', trust_remote_code=True)
gptq_model.quantize(CALIBRATION_TEXTS); gptq_model.save_quantized(str(gptq_path)); tokenizer.save_pretrained(str(gptq_path))
del gptq_model; gc.collect(); torch.cuda.empty_cache()
print(f'GPTQ saved to {gptq_path}')

In [ ]:
# AWQ 4-bit
from awq import AutoAWQForCausalLM
awq_path = OUTPUT_DIR / 'awq-4bit'
awq_model = AutoAWQForCausalLM.from_pretrained(str(MODEL_PATH), device_map='auto', trust_remote_code=True)
awq_model.quantize(tokenizer, {'zero_point': True, 'q_group_size': 128, 'w_bit': 4, 'version': 'GEMM'}, calib_data=CALIBRATION_TEXTS)
awq_model.save_quantized(str(awq_path)); tokenizer.save_pretrained(str(awq_path))
del awq_model; gc.collect(); torch.cuda.empty_cache()
print(f'AWQ saved to {awq_path}')

In [ ]:
# Identical full evaluation after every quantization method. GPTQ/AWQ already contain quantization config.
from eval.run import run_full_eval

for name, model_path in {'gptq_4bit': gptq_path, 'awq_4bit': awq_path}.items():
    results = run_full_eval(model_path=str(model_path), data_dir=str(DATA_DIR), judge_model=JUDGE_MODEL, output_dir=str(OUTPUT_DIR / f'eval_{name}'), load_in_4bit=False)
    for split, result in results.items():
        wandb.log({f'{name}/{split}/exact_match': result.metrics.get('exact_match', 0), f'{name}/{split}/rougeL_f1': result.metrics.get('rougeL', {}).get('rougeL_f1', 0), f'{name}/{split}/bleu': result.metrics.get('bleu', 0)})
        for category, scores in result.judge_scores.items():
            wandb.log({f'{name}/{split}/judge/{category}/{metric}': value for metric, value in scores.items()})
        if result.adversarial:
            wandb.log({f'{name}/{split}/{metric}': value for metric, value in result.adversarial.items() if isinstance(value, (int, float))})
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
artifact = wandb.Artifact('models-gptq-awq-4bit', type='model', metadata=config); artifact.add_dir(str(OUTPUT_DIR)); wandb.log_artifact(artifact)
wandb.finish(); print(f'Published Kaggle output: {OUTPUT_DIR}')